In [4]:
import pickle
from FlyOutput import FlyOutput
import Plotters
import plotly.graph_objects as go
import numpy as np  
import Utils
import matplotlib.pyplot as plt
%matplotlib qt

frame_num = 370
cam = 0
image_path = 'D:/Documents/data_for_gs/mov1_2023_08_09_60ms/'
dict_path  = 'D:/Documents/data_for_gs/mov1_2023_08_09_60ms/dict/frames_model.pkl'
path_output = 'D:/Documents/gaussian_model_output/'
interest_point_h5_path = 'G:/My Drive/Research/gaussian_splatting/gaussian_splatting_input/evalutation'

model_name = 'fly_features_compare'
file_name = 'fly_model'


model_name = 'fly_features_dense'
file_name = 'fly_model'
# model_name = 'fly_features_3cam'
# file_name = 'fly_model_aa'
iteration = 1200
frame0 = 370
input_dir = f'{path_output}/{model_name}'



with open(f'{input_dir}/{file_name}_results.pkl', 'rb') as handle:
    output_angles_weights = pickle.load(handle)
    

with open(dict_path,'rb') as f:
    frames = pickle.load(f)


In [5]:

def intersect_all_cams(frames,cam,intersected,tol = 1):
    for cam in range(4):
        intersected = Utils.intersection_per_cam(frames, cam, intersected, tol=tol) 
    return intersected

frames_list=[]

interest_points_dict = {'left_wing_boundry': [0,1,2,3,4,5], 'left_center': [6], 'left_root' : [7],
                              'right_wing_boundry': [8,9,10,11,12,13], 'right_center': [14], 'right_root' : [15],
                              'body_head': [16],'body_tail': [17]}

interest_points = {'left_wing_boundry': [0,1,2,3,4,5], 'left_root' : [7],
                              'right_wing_boundry': [8,9,10,11,12,13], 'right_root' : [15],
                              'body_head': [16],'body_tail': [17]}

for frame_num in range(370,470):
    frame = FlyOutput(image_path,frame_num,input_dir,output_angles_weights,frame0,iteration,file_name,frames_dict = frames)
    frame.intersect_projections  
    frame.interest_load_and_intersect(interest_point_h5_path,num_of_bins = 20)
    frame.calc_wing_le_te(num_of_bins = 20)
    frame.wings_interest_point( left_wing = interest_points_dict['left_wing_boundry'] + interest_points_dict['left_root'],right_wing = interest_points_dict['right_wing_boundry'] + interest_points_dict['right_root'] )
    frame.get_all_interest_2d_projection(interest_points = np.hstack(interest_points.values()))
    frames_list.append(frame)

prev_length = 0 
frames_interest_points = {}
for key in interest_points.keys():
    frames_interest_points[key] = list(range(prev_length,prev_length + len(interest_points[key]))) 
    prev_length += len(interest_points[key])



d:\Documents\model_gaussian_splatting\evaluation\FlyOutput.py:200: FutureWarning:

arrays to stack must be passed as a "sequence" type such as list or tuple. Support for non-sequence iterables such as generators is deprecated as of NumPy 1.16 and will raise an error in the future.

C:\Users\Roni\AppData\Local\Temp\ipykernel_18512\3864591785.py:22: FutureWarning:

arrays to stack must be passed as a "sequence" type such as list or tuple. Support for non-sequence iterables such as generators is deprecated as of NumPy 1.16 and will raise an error in the future.



In [ ]:
# plot error histograms 
top_perc_ol = 0.03
points_to_plot = np.hstack(frames_interest_points.values())
body_points = [14,15]
points_to_plot_rwing = frames_interest_points['right_wing_boundry'] + frames_interest_points['right_root']
points_to_plot_lwing = frames_interest_points['left_wing_boundry'] + frames_interest_points['left_root']


hist_points = Utils.stack_filter_hist_all_2d(frames_list,points_to_plot,top_perc_ol)
hist_points_3d = Utils.stack_filter_hist_all_3d(frames_list,top_perc_ol)
wing_histogram_data_3d = Utils.stack_filter_hist_points_3d(frames_list, top_perc_ol,points_to_plot_rwing,points_to_plot_lwing)
wing_histogram_data_2d = Utils.stack_filter_hist_points_2d(frames_list, top_perc_ol,points_to_plot_rwing,points_to_plot_lwing)



Plotters.plot_cameras_points_hist(2,2,hist_points)
plt.figure()
plt.title(f'All points - Reprojection error mean {np.mean(hist_points.flatten()):.2f} std{np.std(hist_points.flatten()):.2f}')
plt.hist(hist_points.flatten())
plt.xlabel(f'Reprojection error [pixels]')


plt.figure()
plt.title(f'All points - 3D distance mean {np.mean(hist_points_3d.flatten()*1000):.2f} std{np.std(hist_points_3d.flatten()*1000):.2f}')
plt.hist(1000*hist_points_3d.flatten())
plt.xlabel(f'3D distance [mm]')


Plotters.plot_body_hist(2,1,body_points,hist_points,['body tail','body head'])
Plotters.plot_subplot_hist_wing(4,2,wing_histogram_data_3d)
Plotters.plot_subplot_hist_wing(4,2,wing_histogram_data_2d)




C:\Users\Roni\AppData\Local\Temp\ipykernel_18512\3950228698.py:3: FutureWarning:

arrays to stack must be passed as a "sequence" type such as list or tuple. Support for non-sequence iterables such as generators is deprecated as of NumPy 1.16 and will raise an error in the future.



In [19]:
import plotly.graph_objects as go
import numpy as np


color_list = ['lime','crimson','magenta','magenta','dodgerblue','blue','blue','black','orange']
name_list = ['body','right wing','right wing le','right wing te','left wing','left wing le','left wing te','Amitai points','gaussian points']
size_list = [2,2,4,4,2,4,4,5,5]
framestart = 370
frame_end = 470
frames = range(framestart, frame_end)
output_path = f'{path_output}/{model_name}/animated_plot.html'

xyz_all_frames = np.vstack([frame.xyz_rotated for frame in frames_list])

# === HELPERS ===

def create_scatter3d(xyz, color,name,size = 2):
    """Create a single 3D scatter trace for a specific part."""
    return go.Scatter3d(
        x=xyz[:, 0],
        y=xyz[:, 1],
        z=xyz[:, 2],
        mode="markers",
        name = name,
        marker=dict(size=size, opacity=1, color=color, colorscale='gray'),
    )


def get_global_bounds(xyz_list):
    """Compute global min and max coordinates over all frames for consistent axis scaling."""
    return np.min(xyz_list, axis=0), np.max(xyz_list, axis=0)


def create_frame(parts_list, color_list,size_list, frame_name,name_list):
    """Create one animation frame with all parts for a given timestep."""
    data = [
        create_scatter3d(part, color,name,size)
        for part, color,size,name in zip(parts_list, color_list, size_list,name_list)
    ]
    return go.Frame(data=data, name=frame_name)


def create_play_pause_buttons():
    """Return Play/Pause button definitions for animation."""
    return [
        {
            "buttons": [
                {
                    "args": [None, {"frame": {"duration": 100, "redraw": True}, "fromcurrent": True}],
                    "label": "Play",
                    "method": "animate",
                },
                {
                    "args": [[None], {"frame": {"duration": 0, "redraw": True}, "mode": "immediate"}],
                    "label": "Pause",
                    "method": "animate",
                },
            ],
            "direction": "left",
            "pad": {"r": 10, "t": 87},
            "showactive": False,
            "type": "buttons",
            "x": 0.1,
            "xanchor": "right",
            "y": 0,
            "yanchor": "top",
        }
    ]


def create_slider(frames_range):
    """Create a slider object to control the animation."""
    return [
        {
            "active": 0,
            "steps": [
                {
                    "args": [[str(i)], {"frame": {"duration": 100, "redraw": True}, "mode": "immediate"}],
                    "label": str(frame),
                    "method": "animate",
                }
                for i, frame in enumerate(frames_range)
            ],
        }
    ]


# === MAIN FUNCTION ===

def create_3d_animation(frames_list, color_list,xyz_all_frames,size_list,name_list):
    """Build and show the 3D animation."""
    min_xyz, max_xyz = get_global_bounds(xyz_all_frames)

    # Initial frame data
    intial_parts = [frames_list[0].body,frames_list[0].right_wing,frames_list[0].right_wing_le,frames_list[0].right_wing_te,frames_list[0].left_wing,frames_list[0].left_wing_le,frames_list[0].left_wing_te, frames_list[0].rotated_points_3d, frames_list[0].gaussian_closest_to_interest]
    initial_data = [
        create_scatter3d(part, color,name,size)
        for  part,color,size,name in zip(intial_parts, color_list,size_list,name_list)
    ]
    bounding_box_trace = go.Scatter3d(
    x=[min_xyz[0], max_xyz[0]],
    y=[min_xyz[1], max_xyz[1]],
    z=[min_xyz[2], max_xyz[2]],
    mode='markers',
    marker=dict(size=0.1, color='rgba(0,0,0,0)'),
    showlegend=False
    )
    initial_data.append(bounding_box_trace)
    # Create frames for animation
    frames_data = [
        create_frame([xyz_frame.body,xyz_frame.right_wing,xyz_frame.right_wing_le,xyz_frame.right_wing_te,xyz_frame.left_wing,xyz_frame.left_wing_le,xyz_frame.left_wing_te, xyz_frame.rotated_points_3d, xyz_frame.gaussian_closest_to_interest], color_list,size_list, str(i),name_list)
        for i, xyz_frame in enumerate(frames_list[framestart - frame0:frame_end - frame0])
    ]

    # Build full figure
    fig = go.Figure(
        data=initial_data,
        layout=go.Layout(
            scene=dict(
                xaxis_title="X",
                yaxis_title="Y",
                zaxis_title="Z",
            ),
            updatemenus=create_play_pause_buttons(),
            sliders=create_slider(frames),
        ),
        frames=frames_data,
    )

    fig.show()
    fig.write_html(output_path)
    print(f"Saved animation to: {output_path}")


create_3d_animation(frames_list, color_list,xyz_all_frames,size_list,name_list)




    # point_3d_per_frame.append(np.vstack(points_3d))
    # gaussians_interest_points.append(gaussian_points)

Saved animation to: D:/Documents/gaussian_model_output//fly_features_dense/animated_plot.html


In [ ]:
frame = 372
color = frames_list[frame - frame0].color
idx_part = frames_list[frame - frame0].idx_parts


grayscale = (color[:,0] - color[:,0].min()) / (color[:,0].max() - color[:,0].min())

opacity = frames_list[frame - frame0].opacity * grayscale
# grayscale = grayscale[grayscale <1]

frame = frames_list[frame-frame0]
fig = go.Figure()
Plotters.scatter3d(fig,frame.body,'green',3,'body',show_colorbar = False)
Plotters.scatter3d(fig,frame.right_wing,opacity[idx_part[1]],2,'right wing',show_colorbar = False)
Plotters.scatter3d(fig,frame.left_wing,opacity[idx_part[2]],2,'left wing',show_colorbar = False)
Plotters.scatter3d(fig,frame.rotated_points_3d[7:8,:],'blue',5,'interest',show_colorbar = False)
# Plotters.scatter3d(fig,frames_list[frame-frame0].gaussian_closest_to_interest,'orange',5,'interest gauss',show_colorbar = False)
Plotters.scatter3d(fig,frame.right_wing_le,'purple',3,'interest gauss',show_colorbar = False)
Plotters.scatter3d(fig,frame.right_wing_te,'purple',3,'interest gauss',show_colorbar = False)

Plotters.scatter3d(fig,frame.left_wing_le,'purple',3,'interest gauss',show_colorbar = False)
Plotters.scatter3d(fig,frame.left_wing_te,'purple',3,'interest gauss',show_colorbar = False)
Plotters.scatter3d(fig,frame.gaussian_closest_to_interest,'orange',5,'interest gauss',show_colorbar = False)
Plotters.scatter3d(fig,frame.body_interest_gaussian,'orange',5,'interest gauss',show_colorbar = False)

# Plotters.scatter3d(fig,frame.rotated_points_3d[8:16],'orange',3,'interest gauss',show_colorbar = False)
Plotters.scatter3d(fig, np.vstack((np.mean(frame.bottom,axis = 0) - frame.xbody*2/1000,np.mean(frame.body,axis = 0) + frame.xbody*2/1000)),'black',5,'x',mode = 'markers+lines') 
# Plotters.scatter3d(fig,bod_ax_top,'black',3,'body',show_colorbar = False)

Plotters.scatter3d(fig, np.atleast_2d(frame.interest_on_xbody),'black',10,'inter_on_body',mode = 'markers+lines') 
Plotters.scatter3d(fig, frame.rotated_points_3d[16:,:],'black',10,'inter',mode = 'markers+lines') 


fig.show()

ax = None
ax = Plotters.plot_projections(frame.interest_points_3d[:,:],frame.frames,color = 'magenta',ax = ax, size = 5)
ax = Plotters.plot_projections(frame.gaussian_closest_to_interest_ew[:,:],frame.frames,color = 'orange',ax = ax, size = 5)
# ax = Plotters.plot_projections(frame.body_interest_gaussian_ew[:,:],frame.frames,color = 'orange',ax = ax, size = 5)

ax = None
ax = Plotters.plot_projections(frame.right_wing_ew[:,:],frame.frames,color = 'red',ax = ax, size = 5)
ax = Plotters.plot_projections(frame.left_wing_ew[:,:],frame.frames,color = 'blue',ax = ax, size = 5)


In [ ]:
origin_plane = np.mean(frame.body,axis = 0) + frame.xbody*2/1000
vector = frame.body - origin_plane
dist = np.dot(vector,frame.xbody)
projected = frame.body  - dist[:,np.newaxis]*frame.xbody
fig = go.Figure()

Plotters.scatter3d(fig,projected,'orange',5,'interest gauss',show_colorbar = False)
fig.show()
from collections import defaultdict
bin_dict = defaultdict(list)
bin_dict_point = defaultdict(list)

hist, bin_edges = np.histogramdd(projected, bins=10)
ix = np.digitize(points[:, 0], bin_edges[0]) - 1
iy = np.digitize(points[:, 1], bin_edges[1]) - 1
iz = np.digitize(points[:, 2], bin_edges[2]) - 1
for d,point, i, j, k in zip(dist,points, ix, iy, iz):
    bin_dict[(i, j, k)].append(d)
    bin_dict_point[(i, j, k)].append(point)

points_top = []
for distn,pts in zip(bin_dict.values(),bin_dict_point.values()):
    min_idx = np.argmin(distn)
    points_top.append(pts[min_idx])

In [ ]:
from math import atan2

def argsort(seq):
    #http://stackoverflow.com/questions/3382352/equivalent-of-numpy-argsort-in-basic-python/3382369#3382369
    #by unutbu
    #https://stackoverflow.com/questions/3382352/equivalent-of-numpy-argsort-in-basic-python 
    # from Boris Gorelik
    return sorted(range(len(seq)), key=seq.__getitem__)

def rotational_sort(list_of_xy_coords, centre_of_rotation_xy_coord, clockwise=True):
    cx,cy=centre_of_rotation_xy_coord
    angles = [atan2(x-cx, y-cy) for x,y in list_of_xy_coords]
    indices = argsort(angles)
    # if clockwise:
    #     return [list_of_xy_coords[i] for i in indices]
    # else:
    #     return [list_of_xy_coords[i] for i in indices[::-1]]
    return indices


wing_bound = np.vstack((frame.right_wing_le,frame.right_wing_te))

indices = rotational_sort(wing_bound[:,0:2], np.mean(wing_bound[:,0:2],axis = 0), clockwise=True)


wing_bound
fig = go.Figure()
Plotters.scatter3d(fig,frame.right_wing,'red',3,'body',show_colorbar = False)
Plotters.scatter3d(fig,wing_bound[indices,:],'black',3,'body',show_colorbar = False,mode='markers+lines')

fig.show()
